# Metrics and comparison of YOLOv26 models

This notebook creates the YOLOv26 comparison charts from the artifacts stored in `models/`, rather than from manually entered metric values.

- **Global mAP@0.5:** best value in each variant's `results.csv`.
- **mAP@0.5 by class:** a fresh validation of each stored `weights/best.pt` checkpoint against the repository dataset.

Run from the `notebooks/` directory or adjust `PROJECT_ROOT`.

## 1. Load recorded training metrics

The global comparison reads the best recorded validation mAP@0.5 from each model artifact.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

PROJECT_ROOT = Path("..")
MODELS_DIR = PROJECT_ROOT / "models"
DATA_YAML = PROJECT_ROOT / "dataset" / "data.yaml"
VARIANTS = ("n", "s", "m", "l")
MODEL_LABELS = [f"YOLOv26{variant}" for variant in VARIANTS]
METRIC_COLUMN = "metrics/mAP50(B)"

def artifact_path(variant, filename):
    path = MODELS_DIR / f"yolov26{variant}" / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing model artifact: {path}")
    return path

def best_map50(variant):
    with artifact_path(variant, "results.csv").open(newline="") as file:
        rows = list(csv.DictReader(file))
    if not rows or METRIC_COLUMN not in rows[0]:
        raise KeyError(f"{METRIC_COLUMN} is missing from YOLOv26{variant} results.csv")
    return max(float(row[METRIC_COLUMN]) for row in rows)

map50 = [best_map50(variant) for variant in VARIANTS]
print(dict(zip(MODEL_LABELS, map50)))

## 2. Global mAP@0.5 per variant

In [ ]:
colors = ["#F4A261", "#2A9D8F", "#4C78A8", "#B565A7"]
hatches = ["-", "//", "xx", "\\"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(MODEL_LABELS, map50, color=colors, edgecolor="black")
for bar, hatch, value in zip(bars, hatches, map50):
    bar.set_hatch(hatch)
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f"{value:.4f}", ha="center", va="bottom", fontsize=9)

ax.set_title("mAP@0.5 Comparison - YOLOv26 Models")
ax.set_ylabel("mAP@0.5")
ax.set_xlabel("YOLOv26 model")
ax.set_ylim(0, 1.0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
fig.tight_layout()
plt.show()

## 3. mAP@0.5 per class

This cell re-evaluates the stored best checkpoint for each variant. It may take several minutes, but keeps the per-class chart synchronized with the actual weights and dataset.

In [ ]:
classes = ["cat", "civilian", "cow", "dog", "horse", "rescuer"]
per_class_map50 = {}

for variant in VARIANTS:
    checkpoint = artifact_path(variant, "weights/best.pt")
    metrics = YOLO(str(checkpoint)).val(data=str(DATA_YAML), imgsz=640, verbose=False)
    all_ap = np.asarray(metrics.box.all_ap)
    if all_ap.shape != (len(classes), 10):
        raise ValueError(f"Expected {len(classes)} x 10 AP values, received {all_ap.shape} for YOLOv26{variant}")
    per_class_map50[variant] = all_ap[:, 0]  # IoU = 0.5

x = np.arange(len(classes))
width = 0.2
fig, ax = plt.subplots(figsize=(11, 4.5))
for index, (variant, color, hatch) in enumerate(zip(VARIANTS, colors, hatches)):
    ax.bar(
        x + (index - 1.5) * width,
        per_class_map50[variant],
        width,
        label=f"YOLOv26 {variant.upper()}",
        color=color,
        edgecolor="black",
        hatch=hatch,
    )

ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylabel("mAP@0.5")
ax.set_xlabel("Class")
ax.set_ylim(0, 1.0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=4, frameon=False)
fig.tight_layout()
plt.show()